## Exercise 1

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 7: Convolutional Neural Networks (CNN)
# Niveau: Anfänger
# Aufgabe 1 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. MNIST-Daten laden und vorbereiten ─────────────────────────────────────
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# CNN erwartet Kanal-Dimension: (N, 28, 28, 1)
x_train = x_train.astype("float32")[..., np.newaxis] / 255.0
x_test  = x_test.astype("float32")[..., np.newaxis]  / 255.0
print(f"Trainingsdaten: {x_train.shape}  Testdaten: {x_test.shape}")

# ── 2. CNN-Architektur aufbauen ───────────────────────────────────────────────
modell = tf.keras.Sequential([
    # Erster Faltungsblock: 32 Filter, 3×3 Kernel, ReLU
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu",
                           input_shape=(28, 28, 1), name="conv_1"),
    tf.keras.layers.MaxPooling2D((2, 2), name="pool_1"),

    # Zweiter Faltungsblock: 64 Filter, 3×3 Kernel, ReLU
    tf.keras.layers.Conv2D(64, (3, 3), activation="relu", name="conv_2"),
    tf.keras.layers.MaxPooling2D((2, 2), name="pool_2"),

    # Flachdrücken und dichte Schichten
    tf.keras.layers.Flatten(name="flatten"),
    tf.keras.layers.Dense(128, activation="relu", name="dense_1"),
    tf.keras.layers.Dense(10,  activation="softmax", name="ausgabe"),
], name="MNIST_CNN")

# ── 3. Modell kompilieren ─────────────────────────────────────────────────────
modell.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
modell.summary()

# ── 4. Training (5 Epochen) ───────────────────────────────────────────────────
print("\nStarte Training (5 Epochen)...")
history = modell.fit(
    x_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

# ── 5. Evaluation ─────────────────────────────────────────────────────────────
test_loss, test_acc = modell.evaluate(x_test, y_test, verbose=0)
print(f"\nTest-Verlust:     {test_loss:.4f}")
print(f"Test-Genauigkeit: {test_acc:.4f}")

# ── 6. Trainingsverlauf plotten ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["loss"],     label="Training-Loss")
axes[0].plot(history.history["val_loss"], label="Validierungs-Loss")
axes[0].set_title("Verlust (Loss)")
axes[0].set_xlabel("Epoche")
axes[0].set_ylabel("Verlust")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history["accuracy"],     label="Training")
axes[1].plot(history.history["val_accuracy"], label="Validierung")
axes[1].set_title("Genauigkeit (Accuracy)")
axes[1].set_xlabel("Epoche")
axes[1].set_ylabel("Genauigkeit")
axes[1].legend()
axes[1].grid(True)

plt.suptitle("CNN auf MNIST – Trainingsverlauf", fontsize=14)
plt.tight_layout()
plt.savefig("A7_1_cnn_training.png", dpi=100)
plt.show()

# ── 7. Beispielvorhersagen anzeigen ───────────────────────────────────────────
vorhersagen = modell.predict(x_test[:5], verbose=0)
klassen     = np.argmax(vorhersagen, axis=1)

fig, axes = plt.subplots(1, 5, figsize=(12, 2.5))
for i in range(5):
    axes[i].imshow(x_test[i, :, :, 0], cmap="gray")
    farbe = "green" if klassen[i] == y_test[i] else "red"
    axes[i].set_title(f"Pred: {klassen[i]}\nWahr: {y_test[i]}", color=farbe)
    axes[i].axis("off")

plt.suptitle("CNN MNIST – Beispielvorhersagen (grün=korrekt)", fontsize=12)
plt.tight_layout()
plt.savefig("A7_1_cnn_vorhersagen.png", dpi=100)
plt.show()
print("Diagramme gespeichert: A7_1_cnn_training.png, A7_1_cnn_vorhersagen.png")


## Exercise 2

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 7: Convolutional Neural Networks (CNN)
# Niveau: Anfänger
# Aufgabe 2 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
from scipy.ndimage import convolve
import matplotlib

import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# ── 1. Ein MNIST-Bild laden ───────────────────────────────────────────────────
(x_train, y_train), _ = tf.keras.datasets.mnist.load_data()
bild = x_train[0].astype("float32")  # 28×28 Pixel
print(f"Bild-Shape: {bild.shape}, Ziffer: {y_train[0]}")

# ── 2. Filter definieren ──────────────────────────────────────────────────────

# Filter 1: Kantenerkennung (Sobel-ähnlich, horizontal)
filter_kanten = np.array([
    [-1, -1, -1],
    [ 0,  0,  0],
    [ 1,  1,  1]
], dtype="float32")

# Filter 2: Weichzeichner (Gauss-ähnlich, Durchschnitt)
filter_weich = np.array([
    [1/9, 1/9, 1/9],
    [1/9, 1/9, 1/9],
    [1/9, 1/9, 1/9]
], dtype="float32")

# Filter 3: Schärfefilter (Unsharp Mask)
filter_schaerfen = np.array([
    [ 0, -1,  0],
    [-1,  5, -1],
    [ 0, -1,  0]
], dtype="float32")

filter_liste = [
    ("Kantenerkennung\n(Sobel horizontal)", filter_kanten),
    ("Weichzeichner\n(Durchschnitt 3×3)",   filter_weich),
    ("Schärfefilter\n(Unsharp Mask)",        filter_schaerfen),
]

# ── 3. Faltung (Convolution) anwenden ────────────────────────────────────────
ergebnisse = []
for name, f in filter_liste:
    gefaltetes_bild = convolve(bild, f, mode="constant", cval=0.0)
    ergebnisse.append((name, gefaltetes_bild))
    print(f"Filter '{name.split(chr(10))[0]}': "
          f"Min={gefaltetes_bild.min():.1f}, Max={gefaltetes_bild.max():.1f}")

# ── 4. Visualisierung ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Original
axes[0].imshow(bild, cmap="gray")
axes[0].set_title(f"Original\n(Ziffer: {y_train[0]})")
axes[0].axis("off")

# Gefaltete Bilder
for ax, (name, ergebnis) in zip(axes[1:], ergebnisse):
    ax.imshow(ergebnis, cmap="gray")
    ax.set_title(name, fontsize=10)
    ax.axis("off")

plt.suptitle("Faltungsoperationen (Convolution) auf MNIST-Bild", fontsize=13)
plt.tight_layout()
plt.savefig("A7_2_faltung_visualisiert.png", dpi=100)
plt.show()
print("Diagramm gespeichert: A7_2_faltung_visualisiert.png")

# ── 5. Erklärung der Faltungsoperation ───────────────────────────────────────
print("\n── Erklärung der Faltungsoperation ──")
print("1. Ein Filter (Kernel) gleitet über das Bild.")
print("2. An jeder Position wird das elementweise Produkt berechnet.")
print("3. Die Summe der Produkte ergibt den neuen Pixelwert.")
print("4. Der Filter ist klein (z.B. 3×3), erkennt aber lokale Muster.")
print()
print("Kantenerkennung: Hebt horizontale Kanten hervor (helle/dunkle Übergänge)")
print("Weichzeichner:   Mittelt benachbarte Pixel → Rauschen wird reduziert")
print("Schärfefilter:   Verstärkt Kontraste → Details werden betont")


## Exercise 3

**Dataset Used:** MNIST (keras.datasets)

The following code implements the steps for this exercise. Outputs and charts are generated automatically inline.

In [ ]:
%matplotlib inline
# ============================================================
# educx GmbH – Neuronale Netze | Modul 2
# Lerntag 7: Convolutional Neural Networks (CNN)
# Niveau: Anfänger
# Aufgabe 3 von 3
# ============================================================
# Musterlösung – lauffähig in Spyder (tf_arm conda env)
# Python-Pfad: /Users/solusprime/opt/anaconda3/envs/tf_arm/bin/python
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import math

print("TensorFlow Version:", tf.__version__)

# ── 1. MNIST laden und CNN trainieren ─────────────────────────────────────────
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train[:5000].astype("float32")[..., np.newaxis] / 255.0
x_test  = x_test.astype("float32")[..., np.newaxis]         / 255.0

modell = tf.keras.Sequential([
    tf.keras.layers.Conv2D(16, (3, 3), activation="relu",
                           input_shape=(28, 28, 1), padding="same", name="conv_1"),
    tf.keras.layers.MaxPooling2D((2, 2), name="pool_1"),
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu",
                           padding="same", name="conv_2"),
    tf.keras.layers.MaxPooling2D((2, 2), name="pool_2"),
    tf.keras.layers.Flatten(name="flatten"),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
], name="Feature_Map_CNN")

modell.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
print("Trainiere kleines CNN...")
modell.fit(x_train, y_train[:5000], epochs=3, batch_size=64,
           validation_split=0.1, verbose=1)
print("Training abgeschlossen.")

# ── 2. Modell für Feature-Map-Extraktion erstellen ────────────────────────────
# Zwischenmodell: gibt Ausgabe der ersten Conv-Schicht zurück
feature_map_modell = tf.keras.Model(
    inputs=modell.input,
    outputs=modell.get_layer("conv_1").output,
    name="Feature_Map_Extraktor"
)

# ── 3. Feature Maps für ein Beispielbild extrahieren ─────────────────────────
# Erstes Testbild
probe_bild = x_test[0:1]           # Shape: (1, 28, 28, 1)
feature_maps = feature_map_modell.predict(probe_bild, verbose=0)
print(f"\nFeature-Map-Shape (conv_1): {feature_maps.shape}")
# → (1, 28, 28, 16): 16 verschiedene Feature Maps für das eine Bild

# ── 4. Feature Maps visualisieren ────────────────────────────────────────────
n_filter = feature_maps.shape[-1]     # = 16
n_spalten = 4
n_zeilen  = math.ceil(n_filter / n_spalten)

fig, axes = plt.subplots(n_zeilen + 1, n_spalten, figsize=(n_spalten * 3, (n_zeilen + 1) * 3))

# Originalbild in der ersten Zeile
for col in range(n_spalten):
    if col == 0:
        axes[0, col].imshow(probe_bild[0, :, :, 0], cmap="gray")
        axes[0, col].set_title(f"Originalbild\n(Ziffer: {y_test[0]})", fontsize=9)
    else:
        axes[0, col].axis("off")
axes[0, 0].axis("off")
axes[0, 0].imshow(probe_bild[0, :, :, 0], cmap="gray")

# Feature Maps in den weiteren Zeilen
for i in range(n_filter):
    zeile  = i // n_spalten + 1
    spalte = i  % n_spalten
    fm = feature_maps[0, :, :, i]
    axes[zeile, spalte].imshow(fm, cmap="viridis")
    axes[zeile, spalte].set_title(f"Filter {i+1}", fontsize=8)
    axes[zeile, spalte].axis("off")

# Leere Subplots ausblenden
for i in range(n_filter, n_zeilen * n_spalten):
    zeile  = i // n_spalten + 1
    spalte = i  % n_spalten
    axes[zeile, spalte].axis("off")

plt.suptitle("Feature Maps der ersten Conv2D-Schicht (16 Filter auf MNIST)", fontsize=13)
plt.tight_layout()
plt.savefig("A7_3_feature_maps.png", dpi=100)
plt.show()
print("Diagramm gespeichert: A7_3_feature_maps.png")

# ── 5. Statistiken der Feature Maps ──────────────────────────────────────────
print(f"\nStatistiken der 16 Feature Maps:")
for i in range(n_filter):
    fm = feature_maps[0, :, :, i]
    print(f"  Filter {i+1:2d}: min={fm.min():.3f}, max={fm.max():.3f}, "
          f"mittelwert={fm.mean():.3f}")
